A language model never reads text. It reads a sequence of integer IDs, and the tokenizer decides how text is cut into the pieces those IDs stand for. Each ID selects one row of the model's embedding matrix, so the tokenizer is fixed the moment pretraining starts.

This has a practical consequence. When you fine-tune a pretrained model, you must use exactly the tokenizer it was pretrained with: the same vocabulary, the same merge rules and the same special tokens. A different tokenizer of the same kind assigns different IDs to the same text, and the model then looks up unrelated embedding rows. If no pretrained model covers your language, or your text is very different from what it saw, you train a new tokenizer on your own corpus and pretrain the model with it. The [Hugging Face course](https://huggingface.co/learn/llm-course/chapter6/2) shows how.

This section moves from words and characters to the subword tokenizers that current language models use, and ends with what the choice of tokenizer costs a model.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken"], check=True)
import warnings
warnings.filterwarnings("ignore")
from transformers.utils import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars
hf_logging.set_verbosity_error()
_ = disable_progress_bars()

## Word-level tokenization

The simplest tokenizer splits text at spaces and punctuation and gives every distinct word an ID. Any word that was not in the training text maps to a single unknown token, usually written `<unk>`. The model cannot tell one unknown word from another.

In [ ]:
import re

def word_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

train_text = "The dog runs. The cat runs. A dog chased the cat."
vocab = {"<unk>": 0}
for w in word_tokenize(train_text):
    vocab.setdefault(w, len(vocab))

def word_encode(text):
    return [vocab.get(w, 0) for w in word_tokenize(text)]

print(vocab)
print(word_tokenize("The dog is running."), word_encode("The dog is running."))

The word "running" maps to `<unk>` although "runs" is in the vocabulary. A word-level vocabulary does not record that the words run, runs and running share a stem, so each form needs its own entry and its own training examples. Covering a real corpus this way takes hundreds of thousands of entries, and new names, misspellings and technical terms still fall outside it.

## Character-level tokenization

At the other extreme, every character is a token. The vocabulary is small and nothing is ever unknown. The price is sequence length.

In [ ]:
sentence = "Tokenizers decide how a language model sees its input."
print(len(word_tokenize(sentence)), "word tokens")
print(len(sentence), "character tokens")

The same sentence becomes more than five times as many tokens. Self-attention compares every token with every other token, so its cost grows with the square of the sequence length, and a context window of fixed size holds about a fifth as much text. A single character also carries little meaning, so the model has to spend some of its capacity assembling characters into words.

Subword tokenizers sit between the two extremes. Frequent words get a token of their own, and rare words are split into smaller pieces that are still in the vocabulary.

## Byte pair encoding

Byte pair encoding (BPE) was a text compression method before Sennrich et al. adapted it to build subword vocabularies for machine translation [@Sennrich2015-ry]. Training starts with every word split into characters, plus an end-of-word marker, which is written `_` below. Then it repeats two steps:

1. Count every pair of adjacent symbols across the corpus, weighted by how often each word occurs.
2. Merge the most frequent pair into a new symbol, add it to the vocabulary and record the merge.

Training stops when the vocabulary reaches the size you chose. The ordered list of merges is the tokenizer. The trainer below runs on a corpus of one sentence.

In [ ]:
from collections import Counter

def pair_counts(words):
    pairs = Counter()
    for symbols, freq in words.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[a, b] += freq
    return pairs

def apply_merge(symbols, pair):
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)

def train_bpe(corpus, num_merges):
    words = Counter(tuple(w) + ("_",) for w in corpus.split())
    merges = []
    for step in range(1, num_merges + 1):
        pairs = pair_counts(words)
        if not pairs:
            break
        best, count = max(pairs.items(), key=lambda kv: kv[1])
        new_words = Counter()
        for symbols, freq in words.items():
            new_words[apply_merge(symbols, best)] += freq
        words = new_words
        merges.append(best)
        print(f"merge {step:2d}: {best[0]!r:>8} + {best[1]!r:<6} -> {best[0] + best[1]!r:<10} count {count}")
    return merges, words

corpus = "she sells seashells by the seashore"
merges, words = train_bpe(corpus, num_merges=10)

The counts are recomputed after every merge. At the start, `s h`, `h e`, `s e` and `e _` all occur 3 times, and the tie is broken by the order in which the pairs were first seen. Once `s` and `h` merge into `sh`, the pair `h e` survives only in "the", so its count drops from 3 to 1 and it never wins. Counting the pairs once on the starting corpus and merging them in that order gives a different and incorrect sequence. The whole word `she_` appears only at merge 9.

After ten merges, the corpus words look like this:

In [ ]:
for symbols in words:
    print(" ".join(symbols))

To tokenize new text, BPE splits each word into characters and replays the learned merges in the order they were learned. A word the trainer never saw still gets a segmentation, because in the worst case it falls back to single characters.

In [ ]:
def bpe_encode(word, merges):
    symbols = tuple(word) + ("_",)
    for pair in merges:
        symbols = apply_merge(symbols, pair)
    return symbols

for word in ["shells", "she", "sheets", "boy"]:
    print(f"{word:7} -> {bpe_encode(word, merges)}")

### Byte-level BPE in GPT-2

A character-level starting vocabulary has a gap: a character that never appeared in training, such as an emoji or a letter from another alphabet, still has no ID. GPT-2 closes it by running BPE over the UTF-8 bytes of the text instead of its characters [@Radford2019-im]. The starting vocabulary is the 256 possible byte values, so every string can be encoded. On top of them sit 50,000 learned merges and one special end-of-text token, 50,257 entries in all. GPT-3 uses the same tokenizer.

In [ ]:
from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained("openai-community/gpt2")
text = "she sells seashells by the seashore"
tokens = gpt2.tokenize(text)
print(tokens)
print(gpt2.convert_tokens_to_ids(tokens))
print(len(gpt2), "entries in the vocabulary")

The `Ġ` in front of most tokens is a space. Before training, GPT-2 maps each of the 256 byte values to a printable Unicode character, so that the vocabulary and merge files can be stored and read as ordinary text. The 188 bytes that already print as a visible character keep it. The remaining 68, which include space, newline, tab and the other control characters, are assigned the code points from 256 upward in byte order. Space is the 33rd of them, so it lands on code point 288, which is `Ġ`. The function below rebuilds the table.

In [ ]:
def bytes_to_unicode():
    printable = (list(range(ord("!"), ord("~") + 1))
                 + list(range(ord("¡"), ord("¬") + 1))
                 + list(range(ord("®"), ord("ÿ") + 1)))
    table = {b: chr(b) for b in printable}
    n = 0
    for b in range(256):
        if b not in table:
            table[b] = chr(256 + n)
            n += 1
    return table

table = bytes_to_unicode()
for name, b in [("letter a", ord("a")), ("space", 32), ("newline", 10), ("tab", 9), ("delete", 127)]:
    print(f"{name:8} byte {b:3d} -> {table[b]!r} code point {ord(table[b])}")
print(gpt2.tokenize("\n\t"))

The mapping is not a uniform shift by 256: byte 127 lands on 289, right after space. It only exists to make the files printable, and decoding reverses it exactly.

GPT-2 also splits the text with a regular expression before any merge is applied. The pieces are words with their leading space, runs of digits, runs of punctuation and runs of whitespace, and merges never cross a piece boundary. This keeps tokens such as `dog.` and `dog!` out of the vocabulary.

In [ ]:
print(gpt2.backend_tokenizer.pre_tokenizer.pre_tokenize_str("she sells    seashells!\n"))

In a run of spaces, the last space joins the following word and each of the others stays a separate piece. GPT-2 has no merges for runs of spaces, so each extra space becomes a token of its own. This matters for indented code, as the last part of this section shows.

The byte fallback also shows which languages the merges were learned from. GPT-2 was trained mostly on English web text, so an English phrase becomes a few word tokens while a Greek phrase of similar length breaks into pieces of single letters and bytes.

In [ ]:
for s in ["Good morning, world", "Καλημέρα κόσμε"]:
    print(f"{s!r}: {len(s)} characters, {len(gpt2.tokenize(s))} tokens")

## WordPiece

BERT uses WordPiece, which builds its vocabulary by merging pairs like BPE but chooses the pair differently [@Wu2016-kd]. Instead of the most frequent pair, it merges the pair with the highest score

$$\mathrm{score}(a, b) = \frac{\mathrm{count}(ab)}{\mathrm{count}(a)\,\mathrm{count}(b)},$$

which favors pairs whose parts rarely occur apart. Encoding is also different. WordPiece does not replay merges. It takes the longest vocabulary entry that matches the start of the word, then the longest entry that matches what remains, and so on. Pieces that continue a word carry the prefix `##`, which tells the decoder to attach them to the previous piece without a space.

In [ ]:
bert = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
for text in ["I have a new GPU!", "tokenization"]:
    print(bert.tokenize(text))

The `bert-base-uncased` tokenizer lowercases its input before anything else, so "GPU" is looked up as "gpu". That word is not in the vocabulary of 30,522 entries, and the longest match from the left is `gp`, followed by `##u`.

## Unigram and SentencePiece

The [Unigram tokenizer](https://arxiv.org/abs/1804.10959) of Kudo works in the opposite direction. It starts from a large candidate vocabulary and assigns each token a probability. It then repeatedly removes the tokens whose removal reduces the likelihood of the training corpus the least, until the vocabulary has the target size. To tokenize a word, it picks the segmentation with the highest probability.

[SentencePiece](https://arxiv.org/abs/1808.06226) is a library that trains BPE or Unigram tokenizers directly on raw text. It treats the space as an ordinary symbol, written `▁`, so it needs no language-specific rules for splitting words. That is useful for languages such as Japanese and Chinese, which do not separate words with spaces. T5 uses a SentencePiece Unigram model, and Llama 2 a SentencePiece BPE model.

In [ ]:
t5 = AutoTokenizer.from_pretrained("google-t5/t5-small")
print(t5.tokenize("I have a new GPU!"))

## Vocabulary size and sequence length

The [tiktokenizer app](https://tiktokenizer.vercel.app/?model=gpt2) runs several tokenizers in the browser and colors each token. Paste the following program into it and switch between the `gpt2` and `cl100k_base` tokenizers.

In [ ]:
code_sample = '''# A simple Python program to calculate the area of a rectangle

# Function to calculate the area of a rectangle
def calculate_rectangle_area(length, width):
  """
  Calculates the area of a rectangle.

  Args:
    length: The length of the rectangle.
    width: The width of the rectangle.

  Returns:
    The area of the rectangle.
  """
  area = length * width
  return area

# Get input from the user
length = float(input("Enter the length of the rectangle: "))
width = float(input("Enter the width of the rectangle: "))

# Calculate the area
area = calculate_rectangle_area(length, width)

# Print the result
print("The area of the rectangle is:", area)
'''

<iframe
	src="https://tiktokenizer.vercel.app/?model=gpt2"
	frameborder="0"
	width="850"
	height="550"
></iframe>

The comparison below uses three OpenAI tokenizers: `gpt2`, `cl100k_base` (GPT-3.5 and GPT-4) and `o200k_base` (GPT-4o). For each one it counts the tokens of the program, the tokens that contain only whitespace, and the tokens left once the indentation is removed.

In [ ]:
import tiktoken

dedented = "\n".join(line.lstrip() for line in code_sample.splitlines())
print(f"{'tokenizer':12} {'vocab':>8} {'tokens':>7} {'whitespace':>11} {'no indent':>10}")
for name in ["gpt2", "cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(name)
    ids = enc.encode(code_sample)
    blank = sum(1 for i in ids if enc.decode([i]).strip() == "")
    print(f"{name:12} {enc.n_vocab:8,} {len(ids):7} {blank:11} {len(enc.encode(dedented)):10}")

Whitespace accounts for 23 of the 38 extra tokens that `gpt2` needs: it spends 40 tokens on pieces that hold only spaces or newlines, against 17 for `cl100k_base`. The newer tokenizer has merges for runs of spaces, so the indentation of a line costs one token instead of one per space. It also has merges that join a newline to the punctuation before it, such as `.\n`, and to other newlines. One line of the docstring shows both.

In [ ]:
line = "    The area of the rectangle.\n"
for name in ["gpt2", "cl100k_base"]:
    enc = tiktoken.get_encoding(name)
    print(f"{name:12}", [enc.decode([i]) for i in enc.encode(line)])

Removing the indentation only narrows the gap from 38 tokens to 32, because the newlines remain. The rest of the gap comes from the larger vocabulary, which holds more whole words and more common code fragments. Doubling the vocabulary again, from `cl100k_base` to `o200k_base`, saves nothing on this program. The extra entries pay off on other languages, as the Greek sentence below shows.

In [ ]:
greek = "Καλημέρα κόσμε, τι κάνεις σήμερα;"
for name in ["gpt2", "cl100k_base", "o200k_base"]:
    print(f"{name:12} {len(tiktoken.get_encoding(name).encode(greek)):3} tokens for {len(greek)} characters")

Fewer tokens for the same text have three effects. More text fits in a context window of fixed length. Each self-attention layer compares every pair of tokens, so its cost scales with the square of the length, and 187 tokens cost about 1.6 times as much as 149. Text generation is also faster, because the model produces one token per step.

A larger vocabulary has a cost too. The embedding matrix has one row of width $d$ per vocabulary entry, and the output layer that predicts the next token has the same shape. GPT-2 shares a single matrix between the two, and the calculation below assumes GPT-3 does as well.

In [ ]:
models = [("GPT-2 small", 768, 124.4e6), ("GPT-3", 12288, 175e9)]
for model, d, total in models:
    rest = total - 50257 * d  # parameters outside the embedding matrix
    for name in ["gpt2", "cl100k_base"]:
        V = tiktoken.get_encoding(name).n_vocab
        print(f"{model:11} {name:12} V*d = {V * d / 1e6:7.1f}M, {100 * V * d / (rest + V * d):4.1f}% of all parameters")

For GPT-2 small, the embedding matrix is almost a third of the parameters, and switching to the `cl100k_base` vocabulary would add another 38 million. For a model the size of GPT-3, the same matrix is well under one percent of the parameters. At that scale the extra parameters cost little compared with the shorter sequences they buy, which is one reason recent large models use vocabularies of 100,000 to 200,000 entries.

## References

* Sennrich et al., Neural Machine Translation of Rare Words with Subword Units, 2016 [@Sennrich2015-ry].
* Radford et al., Language Models are Unsupervised Multitask Learners, 2019 [@Radford2019-im].
* Wu et al., Google's Neural Machine Translation System: Bridging the Gap between Human and Machine Translation, 2016 [@Wu2016-kd].
* Kudo, Subword Regularization: Improving Neural Network Translation Models with Multiple Subword Candidates, 2018, [arXiv:1804.10959](https://arxiv.org/abs/1804.10959).
* Kudo and Richardson, SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing, 2018, [arXiv:1808.06226](https://arxiv.org/abs/1808.06226).